# Analisis Data

Mengulang (reproduce) contoh yang ada pada halaman Skforecast Explainability dengan menjalankannya sendiri di Google Colab atau Jupyter Notebook lokal.


## Jawab Pertanyaan

### 1. Analisis prediksi tentang apa?

Analisis ini digunakan untuk memprediksi kebutuhan listrik harian (Demand) berdasarkan:

- Riwayat kebutuhan listrik pada beberapa hari sebelumnya (lag)
- Suhu rata-rata harian (Temperature)

Dataset yang digunakan adalah dataset vic_electricity, yang berisi data konsumsi listrik dan temperatur di wilayah Victoria, Australia.

Target yang diprediksi yaitu Nilai Demand (permintaan/konsumsi listrik) pada hari berikutnya.

### 2. Bagaimana bentuk data trainingnya(apa saja input dan outputnya)

Input menggunakan input(x) yang terdiri dari lag,selain itu juga ada variabel eksternal(exogenous) yaitu berupa suhu harian dan output(y) berisi nilai demand pada hari yang akan diprediksi

### 3. Apa itu lag?

Lag adalah nilai historis (masa lalu) dari variabel target yang digunakan sebagai input model.

Pada kode di halaman Skforecast Explainability menggunakan 7 lag yaitu data 7 hari sebelumnya untuk memprediksi hari berikutnya, dan Lag digunakan karena pada data time series, nilai masa lalu biasanya memengaruhi nilai masa depan.

### 4. Jelaskan proses analisis yang dilakukan

#### 1. Mengambil data

    data = fetch_dataset(name="vic_electricity")

Pada tahap ini, dataset vic_electricity diambil dari library Skforecast. Dataset ini berisi data historis konsumsi listrik (Demand) dan suhu (Temperature) di wilayah Victoria, Australia.

Tujuan tahap ini adalah menyediakan data yang akan digunakan untuk melatih dan menguji model prediksi. Data yang diperoleh masih berupa data deret waktu (time series) dengan indeks berupa tanggal dan waktu.

Data utama yang digunakan yaitu:

 - Demand : jumlah kebutuhan atau konsumsi listrik.
 - Temperature : suhu lingkungan pada waktu tertentu.

#### 2. Agregasi data menjadi harian

    data = data.resample('D').agg({
        'Demand': 'sum',
        'Temperature': 'mean'
    })
    data.head(3)

Dataset asli memiliki frekuensi yang lebih kecil dari satu hari (misalnya per jam atau per setengah jam). Oleh karena itu dilakukan proses agregasi menjadi data harian.

sebelum ke agregasi perlu dilakukan mengubah frekuensi data menjadi perhari,setelah itu dilakukan agregasi dengan menjumlahkan seluruh konsumsi listrik dalam 1 harinya sehingga diperoleh total kebutuhan listrik harian. Dilanjut dengan menghitung rata-rata suhu agar hasil agregasi ini membuat data lebih mudah digunakan untuk prediksi kebutuhan listrik harian.

contoh
| Tanggal    | Demand | Temperature |
| ---------- | ------ | ----------- |
| 01-01-2014 | 52000  | 25.3        |
| 02-01-2014 | 54800  | 26.1        |

#### 3. Membagi data training dan testing

    data_train = data.loc[:'2014-12-21']
    data_test = data.loc['2014-12-22':]

Data dibagi menjadi 2 bagian yaitu:

- Data Training : Berisi seluruh data sampai tanggal 21 Desember 2014.

Data ini digunakan untuk melatih model agar mempelajari pola hubungan antara data masa lalu dan data yang akan datang.

- Data Testing : Berisi data mulai 22 Desember 2014 dan seterusnya.

Data ini tidak digunakan saat pelatihan model. Tujuannya adalah untuk menguji kemampuan model dalam memprediksi data yang belum pernah dilihat sebelumnya.

Pembagian ini penting agar evaluasi model lebih objektif dan tidak hanya mengukur kemampuan model mengingat data lama.

#### 4. Membuat Model Forecasting

    forecaster = ForecasterRecursive(
        regressor=LGBMRegressor(random_state=123, verbose=-1),
        lags=7
    )

ForecasterRecursive adalah kelas dari Skforecast yang digunakan untuk melakukan forecasting (peramalan) time series menggunakan model machine learning.
   
Masalahnya, algoritma machine learning seperti LightGBM tidak memahami konsep waktu secara langsung.

Karena itu, ForecasterRecursive bertugas mengubah data time series menjadi format yang bisa dipahami oleh LightGBM.

| lag_1 | lag_2 | lag_3 | lag_4 | lag_5 | lag_6 | lag_7 | Target |
| ----- | ----- | ----- | ----- | ----- | ----- | ----- | ------ |
| 145   | 135   | 140   | 130   | 110   | 120   | 100   | 150    |

#### 5. Melatih model

    forecaster.fit(
        y=data_train['Demand'],
        exog=data_train['Temperature']
    )

Pada tahap ini model mempelajari hubungan antara variabel target dan variabel input.Proses pelatihan ini menghasilkan model yang dapat digunakan untuk memprediksi nilai Demand pada masa mendatang.

#### 6. Feature Importance

    forecaster.get_feature_importances()

Feature Importance digunakan untuk mengetahui fitur mana yang paling berpengaruh dalam proses prediksi.

Misalnya diperoleh hasil:

| Feature     | Importance |
| ----------- | ---------- |
| lag_1       | 0.40       |
| lag_7       | 0.25       |
| Temperature | 0.15       |

Interpretasinya:

- lag_1 merupakan fitur yang paling sering digunakan model dalam membuat keputusan.
- lag_7 juga memiliki pengaruh yang cukup besar.
- Temperature tetap berpengaruh namun lebih kecil dibanding lag.

Tahap ini membantu memahami informasi apa yang paling diperhatikan model.

### 7. Analisis SHAP

    explainer = shap.TreeExplainer(forecaster.regressor)
    shap_values = explainer.shap_values(X_train)

    shap.summary_plot(
        shap_values,
        X_train,
        plot_type="bar"
    )

SHAP (SHapley Additive exPlanations) digunakan untuk menjelaskan alasan di balik prediksi model.

- Membuat SHAP explainer : 
Explainer digunakan untuk menghitung kontribusi masing-masing fitur terhadap hasil prediksi.

- Summary Plot : 
Visualisasi ini menunjukkan fitur paling penting secara keseluruhan.

- Force plot :
Menjelaskan satu prediksi tertentu, Sehingga dapat diketahui alasan mengapa model menghasilkan prediksi tersebut.

- Dependence plot :
Digunakan untuk melihat hubungan antara suatu fitur dan dampaknya terhadap prediksi.

#### 8. Melakukan Prediksi

    predictions = forecaster.predict(
    steps=10,
    exog=data_test['Temperature']
    )

Model digunakan untuk memprediksi kebutuhan listrik selama 10 hari ke depan.

Input yang digunakan:

- hasil lag sebelumnya
- data suhu pada periode yang akan diprediksi

contoh:

| Tanggal    | Prediksi Demand |
| ---------- | --------------- |
| 22-12-2014 | 52.300          |
| 23-12-2014 | 53.100          |   

#### 9. Permutation Importance

    r = permutation_importance(
        estimator=forecaster.regressor,
        X=X_train,
        y=y_train,
        n_repeats=3,
        max_samples=0.5,
        random_state=123
    )

Metode ini mengukur pentingnya fitur dengan cara mengacak nilai suatu fitur.

Langkah-langkah:

a. Model dihitung performanya dalam kondisi normal.
    
b. Nilai Temperature diacak.

c. Performa model dihitung kembali.

#### 10. Pertial Dependence Plot (PDP)

    fig, ax = plt.subplots(figsize=(9,4))

    PartialDependenceDisplay.from_estimator(
        estimator=forecaster.regressor,
        X=X_train,
        features=["Temperature", "lag_1"],
        kind='both',
        ax=ax
    )

    plt.show()

PDP digunakan untuk melihat pengaruh rata-rata suatu fitur terhadap hasil prediksi. Berbeda dengan SHAP yang menjelaskan prediksi individu, PDP menjelaskan perilaku model secara umum.

- Temperature

misal:

Temperature 20°C → Demand 45.000

Temperature 30°C → Demand 55.000

Artinya semakin tinggi suhu, semakin tinggi kebutuhan listrik.

- Lag_1

lag_1 meningkat -> Demand prediksi meningkat

Hal ini menunjukkan bahwa konsumsi listrik hari sebelumnya memiliki hubungan positif dengan konsumsi listrik hari berikutnya.

